## 1. Installation

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes datasets
!pip install -q unsloth

## 2. Imports, configuration, reproductibilité

In [ ]:
import json, math, os, random, re, time, hashlib
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DATA_DIR = Path("outputs/step3_dataset")
OUT_DIR = Path("outputs/step4_qlora"); OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"                # base en 4-bit NF4 chargée à la section 4
dataset_cfg = json.load(open(DATA_DIR / "dataset_config.json", encoding="utf-8"))
MAX_SEQ_LENGTH = int(dataset_cfg["stats"]["max_seq_length"])
rag_prompt = json.load(open(DATA_DIR / "rag_prompt.json", encoding="utf-8"))
ABSTAIN_MARKER = rag_prompt["abstain_marker"]

HP = dict(lora_r=16, lora_alpha=16, lora_dropout=0.0, lr=2e-4, epochs=2, per_device_bs=1, grad_accum=16, warmup_ratio=0.05, weight_decay=0.0)
assert torch.cuda.is_available(), "GPU requis (Colab : Exécution > Modifier le type d'exécution > GPU)"
gpu = torch.cuda.get_device_properties(0)
BF16 = torch.cuda.get_device_capability()[0] >= 8      # T4 (7.5) : bf16 seulement émulé -> fp16 ; torch.cuda.is_bf16_supported() peut répondre True à tort
print(f"GPU : {gpu.name} ({gpu.total_memory / 1e9:.0f} Go) | bf16 : {BF16} | max_seq_length : {MAX_SEQ_LENGTH} | k_context étape 3 : {dataset_cfg['k_context']}")

## 3. Données : chargement, vérifications, masquage du prompt

In [ ]:
def load_jsonl(p): return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
train_raw, val_raw = load_jsonl(DATA_DIR / "train.jsonl"), load_jsonl(DATA_DIR / "val.jsonl")
print(f"train : {len(train_raw)} | val : {len(val_raw)}")
print("types train :", dict(Counter(r["meta"]["kind"] for r in train_raw)), "| val :", dict(Counter(r["meta"]["kind"] for r in val_raw)))
assert not ({r["meta"]["id"] for r in train_raw} & {r["meta"]["id"] for r in val_raw}), "id communs entre train et val"
# Le split de l'étape 3 est fait par QUESTION SOURCE : on vérifie qu'aucune question n'est commune.
q_tr = {r["messages"][1]["content"].split("Question :")[-1].strip() for r in train_raw}
q_va = {r["messages"][1]["content"].split("Question :")[-1].strip() for r in val_raw}
assert not (q_tr & q_va), "même question dans train et val"

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.padding_side = "right"
if tok.pad_token is None: tok.pad_token = tok.eos_token

def encode(rec):
    """-> input_ids, labels (-100 sur le prompt), attention_mask."""
    msgs = rec["messages"]
    # On passe par du texte puis on tokenise : indépendant de la version de transformers (le type retourné par apply_chat_template a changé).
    prompt_txt = tok.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True)
    full_txt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    prompt_ids = tok(prompt_txt, add_special_tokens=False)["input_ids"]
    full_ids = tok(full_txt, add_special_tokens=False)["input_ids"]
    assert full_ids[:len(prompt_ids)] == prompt_ids, "le prompt n'est pas un préfixe de l'exemple complet"
    # Qwen ajoute "\n" après <|im_end|> : on retire ce token final de la cible pour ne pas apprendre un token « hors réponse ».
    if tok.decode(full_ids[-1:]) == "\n": full_ids = full_ids[:-1]
    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    return dict(input_ids=full_ids, labels=labels, attention_mask=[1] * len(full_ids))

enc_train, enc_val = [encode(r) for r in train_raw], [encode(r) for r in val_raw]
lens = np.array([len(e["input_ids"]) for e in enc_train + enc_val])
n_ans = np.array([sum(l != -100 for l in e["labels"]) for e in enc_train])
print(f"longueur (tokens) : moy {lens.mean():.0f} | p95 {np.percentile(lens, 95):.0f} | max {lens.max()} (limite {MAX_SEQ_LENGTH})")
print(f"tokens appris par exemple : moy {n_ans.mean():.0f} ({n_ans.sum() / sum(len(e['input_ids']) for e in enc_train):.1%} des tokens : le reste est du contexte masqué)")
assert lens.max() <= MAX_SEQ_LENGTH, "exemple plus long que max_seq_length : relancer l'étape 3 (mesure) ou augmenter la limite"

e0 = next(e for e, r in zip(enc_train, train_raw) if r["meta"]["kind"] == "answer")
print("\n--- Partie APPRISE d'un exemple (le reste est masqué) ---")
print(tok.decode([t for t in e0["labels"] if t != -100]))
print("--- Partie masquée (fin du prompt) ---")
print(tok.decode(e0["input_ids"][len(e0["input_ids"]) - len([t for t in e0["labels"] if t != -100]) - 25: len(e0["input_ids"]) - len([t for t in e0["labels"] if t != -100])]))

## 4. Chargement du modèle 4-bit et des adaptateurs LoRA

In [ ]:
BACKEND = None
TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
try:
    from unsloth import FastLanguageModel
    model, tok_u = FastLanguageModel.from_pretrained(model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH, dtype=None, load_in_4bit=True)
    model = FastLanguageModel.get_peft_model(model, r=HP["lora_r"], lora_alpha=HP["lora_alpha"], lora_dropout=HP["lora_dropout"], target_modules=TARGETS,
                                             use_gradient_checkpointing="unsloth", random_state=SEED)
    BACKEND = "unsloth"
except Exception as e:
    print(f"[!] Unsloth indisponible ou incompatible ({type(e).__name__}: {str(e)[:150]}) -> repli transformers+peft")
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                             bnb_4bit_compute_dtype=torch.bfloat16 if BF16 else torch.float16)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb, device_map={"": 0})
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model = get_peft_model(model, LoraConfig(r=HP["lora_r"], lora_alpha=HP["lora_alpha"], lora_dropout=HP["lora_dropout"], target_modules=TARGETS, task_type="CAUSAL_LM"))
    BACKEND = "transformers+peft"
print("Backend :", BACKEND)
model.print_trainable_parameters()

## 5. Métriques de comportement (fonctions) et état AVANT entraînement

In [ ]:
SOURCES_RE = re.compile(r"\nSources : (\[\d+\](?:, \[\d+\])*)\s*$")

@torch.no_grad()
def generate(rec, max_new_tokens=300):
    txt = tok.apply_chat_template(rec["messages"][:-1], tokenize=False, add_generation_prompt=True)
    ids = tok(txt, add_special_tokens=False, return_tensors="pt")["input_ids"].to(model.device)
    out = model.generate(input_ids=ids, attention_mask=torch.ones_like(ids), max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()

def score(rec, text):
    kind, gold = rec["meta"]["kind"], rec["meta"]["gold_positions"]
    n_ctx = rec["meta"]["context_chunks"] if isinstance(rec["meta"]["context_chunks"], int) else len(rec["meta"]["context_chunks"])
    abst = ABSTAIN_MARKER in text
    m = SOURCES_RE.search("\n" + text) if not abst else None
    cited = [int(x) for x in re.findall(r"\[(\d+)\]", m.group(1))] if m else []
    format_ok = abst and text.strip().startswith(ABSTAIN_MARKER) or (m is not None and all(1 <= c <= n_ctx for c in cited))
    return dict(id=rec["meta"]["id"], kind=kind, abstained=abst, format_ok=bool(format_ok), cited=cited,
                citation_ok=bool(kind == "answer" and not abst and set(gold) <= set(cited)))

def behaviour_report(recs, label):
    t0 = time.time(); rows, gens = [], {}
    model.eval()
    for r in recs:
        g = generate(r); gens[r["meta"]["id"] + "|" + r["meta"]["kind"]] = g; rows.append(score(r, g))
    df = pd.DataFrame(rows)
    ans, noans = df[df.kind == "answer"], df[df.kind != "answer"]
    rep = {"label": label, "n": len(df), "format_ok": float(df.format_ok.mean()),
           "abstention_rappel": float(noans.abstained.mean()) if len(noans) else None,
           "fausse_abstention": float(ans.abstained.mean()) if len(ans) else None,
           "citation_ok": float(ans.citation_ok.mean()) if len(ans) else None}
    print(f"[{label}] {rep}  ({time.time() - t0:.0f}s)")
    return rep, gens

def base_behaviour(recs):
    """Modèle de base = adaptateur LoRA désactivé (poids LoRA initialisés à zéro de toute façon avant entraînement)."""
    if hasattr(model, "disable_adapter"):
        with model.disable_adapter():
            return behaviour_report(recs, "base (sans adaptateur)")
    return behaviour_report(recs, "base (avant entraînement)")

if BACKEND == "unsloth": FastLanguageModel.for_inference(model)
rep_base, gens_base = base_behaviour(val_raw)
if BACKEND == "unsloth": FastLanguageModel.for_training(model)

## 6. Entraînement

In [ ]:
from transformers import Trainer, TrainingArguments

class ListDataset(torch.utils.data.Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]

def collate(batch):
    m = max(len(b["input_ids"]) for b in batch)
    pad = lambda x, v: x + [v] * (m - len(x))
    return dict(input_ids=torch.tensor([pad(b["input_ids"], tok.pad_token_id) for b in batch]),
                labels=torch.tensor([pad(b["labels"], -100) for b in batch]),
                attention_mask=torch.tensor([pad(b["attention_mask"], 0) for b in batch]))

steps_per_epoch = math.ceil(len(enc_train) / (HP["per_device_bs"] * HP["grad_accum"]))

# Calcul manuel du nombre d'étapes de warmup
total_steps = steps_per_epoch * HP["epochs"]
calculated_warmup_steps = int(total_steps * HP["warmup_ratio"])

args = TrainingArguments(
    output_dir=str(OUT_DIR / "checkpoints"), num_train_epochs=HP["epochs"], per_device_train_batch_size=HP["per_device_bs"],
    per_device_eval_batch_size=1, gradient_accumulation_steps=HP["grad_accum"], learning_rate=HP["lr"], lr_scheduler_type="cosine",
    warmup_steps=calculated_warmup_steps, weight_decay=HP["weight_decay"],
    bf16=False, fp16=True, # <-- Correction : on force le FP16 compatible avec le GPU T4
    logging_steps=1,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=2, load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    optim="paged_adamw_8bit", seed=SEED, report_to="none", remove_unused_columns=False
)

trainer = Trainer(model=model, args=args, train_dataset=ListDataset(enc_train), eval_dataset=ListDataset(enc_val), data_collator=collate)
print(f"{len(enc_train)} exemples | {steps_per_epoch} pas d'optimisation par epoch | batch effectif {HP['per_device_bs'] * HP['grad_accum']}")
t0 = time.time(); train_out = trainer.train(); print(f"Entraînement terminé en {(time.time() - t0) / 60:.1f} min")

In [ ]:
log = pd.DataFrame(trainer.state.log_history)
tr_l = log.dropna(subset=["loss"])[["step", "loss"]]; ev_l = log.dropna(subset=["eval_loss"])[["step", "eval_loss"]]
display(ev_l.round(4))
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(6, 3.2)); plt.plot(tr_l.step, tr_l.loss, label="train loss (par pas)"); plt.scatter(ev_l.step, ev_l.eval_loss, c="C3", label="val loss (par epoch)", zorder=3)
    plt.xlabel("pas d'optimisation"); plt.ylabel("perte (réponse seule)"); plt.legend(); plt.tight_layout(); plt.savefig(OUT_DIR / "loss_curve.png", dpi=130); plt.show()
except Exception as e:
    print("(courbe non tracée :", e, ")")
# Lecture : val_loss qui remonte alors que train_loss baisse = sur-apprentissage ; on garde le meilleur checkpoint (load_best_model_at_end).